In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [15]:
df = pd.read_csv("datasets/coimbatore_pois_features_500m_clean.csv")
df.shape

(4018, 28)

In [16]:
df.head()

,osm_id,osm_type,name,category,subcategory,lat,lon,addr_housenumber,addr_street,addr_suburb,...,transport_500m,roads_500m,major_roads_500m,landuse_residential_500m,landuse_commercial_500m,landuse_industrial_500m,landuse_retail_500m,landuse_mixed_500m,landuse_farmland_500m,landuse_other_500m
0,244513191,node,"sri ramakrishna engineering college, vatamalai...",amenity,college,11.101632,76.965305,NaN,NaN,NaN,...,0,38,0,0,0,0,0,0,0,0
1,246761183,node,central theater,amenity,theatre,11.015784,76.955423,NaN,NaN,NaN,...,9,105,31,4,0,1,0,0,0,0
2,247177690,node,thudiyalur police station,amenity,police,11.082068,76.941241,NaN,mettupalayam-coimbatore road,NaN,...,13,96,13,0,0,0,0,0,0,0
3,247177691,node,"shiva temple, 1500 years old",amenity,place_of_worship,11.084471,76.939729,NaN,NaN,NaN,...,4,83,6,0,0,0,0,0,0,0
4,247177822,node,srec health center,amenity,hospital,11.100545,76.963745,NaN,NaN,NaN,...,0,47,1,0,0,0,0,0,0,0


In [17]:
target_col = "pois_500m"
drop_cols = ["osm_id", "name", "addr_street", "addr_housenumber", "addr_postcode"]
feature_cols = [c for c in df.columns if c not in drop_cols + [target_col]]

categorical_cols = ["category", "subcategory", "business_type", "osm_type", "addr_suburb", "addr_city"]
categorical_cols = [c for c in categorical_cols if c in feature_cols]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

X = df[feature_cols].copy()
for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
y = pd.to_numeric(df[target_col], errors="coerce")

mask = y.notna()
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

((3214, 22), (804, 22))

In [18]:
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot()),
    ]
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", numeric_transformer, numeric_cols),
    ]
)

def eval_model(name, model, X_test, y_test):
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    return {"model": name, "rmse": rmse, "mae": mae, "r2": r2}

In [19]:
lin_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ]
)

lin_model.fit(X_train, y_train)
lin_metrics = eval_model("LinearRegression", lin_model, X_test, y_test)
lin_metrics

{'model': 'LinearRegression',
 'rmse': np.float64(6.528614406018107),
 'mae': 4.802135806063465,
 'r2': 0.9617810031755488}

In [22]:
rf_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )),
    ]
)

rf_model.fit(X_train, y_train)
rf_metrics = eval_model("RandomForestRegressor", rf_model, X_test, y_test)
rf_metrics

{'model': 'RandomForestRegressor',
 'rmse': np.float64(2.1609536492960113),
 'mae': 1.14349087893864,
 'r2': 0.9958127571571749}

In [21]:
pd.DataFrame([lin_metrics, rf_metrics]).sort_values(by="rmse")

,model,rmse,mae,r2
1,RandomForestRegressor,2.160954,1.143491,0.995813
0,LinearRegression,6.528614,4.802136,0.961781
